In [1]:
# Imports

import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import mmread

In [2]:
# Definir rutas para no perderme

DATA_DIR = "../data"
RESULTS_DIR = "../results"
FIGURES_DIR = "../figures"

In [ ]:
# Abrir el archivo en backed porque sino explota el kernel

adata = sc.read_h5ad(
    f"{DATA_DIR}/ICB_dataset/134d34af-cbcd-4837-9310-3d1f83ec6f18.h5ad",
    backed="r"
)

print(adata)
print(adata.shape)
print(adata.obs.columns)

AnnData object with n_obs × n_vars = 355941 × 22779 backed at '../data/ICB_dataset/134d34af-cbcd-4837-9310-3d1f83ec6f18.h5ad'
    obs: 'PMID_donor_id', 'donor_id', 'pre_post', 'author_cell_type', 'author_cell_type_update', 'outcome', 'Combined_outcome', 'Cancer_type_update', 'Study_name', 'Primary_or_met', 'donor_id_pre_post', 'donor_id_outcome', 'donor_id_cell_types', 'donor_id_cell_types_pre_post', 'donor_id_pre_post_outcome', 'tissue_ontology_term_id', 'tissue_type', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'PMID', 'Sample', 'Treatment.or.Mode.of.Action', 'suspension_type', 'sex_ontology_term_id', 'assay_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'nCount_RNA', 'nFeature_RNA', 'Study_name_cancer', 'Cell_type_broad', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'vf_vst_counts_mean', 'vf_vst_counts_variance', 'v

El ICB dataset se cargó en modo backed para evitar problemas de memoria,
ya que contiene 355941 células y 22779 genes. Este recurso integra múltiples
tipos de cáncer, por lo que no podía utilizarse directamente para el objetivo
de este trabajo.

In [4]:
adata.obs["Cancer_type_update"].value_counts()

Cancer_type_update
TNBC     113033
ER+       87648
BCC       76529
ccRCC     29993
HER2+     16017
Mel       13141
MBM       10895
iCCA       5433
HCC        3252
Name: count, dtype: int64

Se inspeccionaron los metadatos disponibles en obs y se identificó la columna
Cancer_type_update como la más adecuada para distinguir entre los diferentes
tipos tumorales incluidos en el dataset.

In [ ]:
# crear una mascara para quedarnos solo con los HCC
mask_hcc = adata.obs["Cancer_type_update"] == "HCC"
print(mask_hcc.sum())

3252


Al explorar los valores de Cancer_type_update, se observó que el dataset
contenía una categoría específica para HCC, con un total de 3252 células.
Dado que este trabajo se centra en hepatocarcinoma, se seleccionó únicamente
este subconjunto para los análisis posteriores.

A partir de una máscara lógica se extrajeron únicamente las células
correspondientes a HCC. Este subconjunto se cargó en memoria y se guardó
como un nuevo archivo .h5ad para facilitar su reutilización posterior.

In [6]:
# Cragar HCC

adata_hcc = adata[mask_hcc, :].to_memory()
print(adata_hcc)
print(adata_hcc.shape)

AnnData object with n_obs × n_vars = 3252 × 22779
    obs: 'PMID_donor_id', 'donor_id', 'pre_post', 'author_cell_type', 'author_cell_type_update', 'outcome', 'Combined_outcome', 'Cancer_type_update', 'Study_name', 'Primary_or_met', 'donor_id_pre_post', 'donor_id_outcome', 'donor_id_cell_types', 'donor_id_cell_types_pre_post', 'donor_id_pre_post_outcome', 'tissue_ontology_term_id', 'tissue_type', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'PMID', 'Sample', 'Treatment.or.Mode.of.Action', 'suspension_type', 'sex_ontology_term_id', 'assay_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'nCount_RNA', 'nFeature_RNA', 'Study_name_cancer', 'Cell_type_broad', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'vf_vst_counts_mean', 'vf_vst_counts_variance', 'vf_vst_counts_variance.expected', 'vf_vst_counts_variance.standardized', 'vf_

In [7]:
adata_hcc.write(f"{DATA_DIR}/adata_ICB_HCC_raw.h5ad")